# 10. Eksport modelu na telefon

**Krok procesu:** 11. Wdrożenie (część po stronie modelu). Aplikacja na Androida to osobny krok.

Format: **ONNX**. Konwerter `skl2onnx` przenosi cały pipeline sklearn (StandardScaler + MLP) do jednego pliku.
Na Androidzie uruchamia go ONNX Runtime. Preprocessing (nadgarstek w (0, 0), skala) to kilka linijek, które
aplikacja liczy sama przed wywołaniem modelu.

Co sprawdzamy:
1. czy model ONNX daje te same wyniki co sklearn na całym zbiorze testowym (zgodność),
2. ile waży i jak szybko liczy jedną dłoń,
3. zapisujemy wektory testowe, żeby aplikacja mogła sprawdzić, że jej preprocessing liczy to samo.

In [1]:
import sys
sys.path.insert(0, "../src")

import json
import pickle
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
from skl2onnx import to_onnx

from pipeline import MODELS, PREPROCESS, load_split

DEPLOY = Path("../deploy")
DEPLOY.mkdir(exist_ok=True)
gates = json.loads(Path("../experiments/gates.json").read_text())
prep_name = gates["final"]["preprocess"]
prep = PREPROCESS[prep_name]
model = pickle.load(open(MODELS / "final.pkl", "rb"))
print("Preprocessing:", prep_name)

Preprocessing: wrist_scale


In [2]:
lm_te, y_te, _ = load_split("test")
X_te = prep(lm_te).astype(np.float32)
onx = to_onnx(model, X_te[:1], options={id(model): {"zipmap": False}}, target_opset=17)
onnx_path = DEPLOY / "finger_counter.onnx"
onnx_path.write_bytes(onx.SerializeToString())
session = ort.InferenceSession(str(onnx_path))
inputs = [(i.name, i.shape, i.type) for i in session.get_inputs()]
outputs = [(o.name, o.shape, o.type) for o in session.get_outputs()]
print("wejścia:", inputs); print("wyjścia:", outputs)

wejścia: [('X', [None, 42], 'tensor(float)')]
wyjścia: [('label', [None], 'tensor(int64)'), ('probabilities', [None, 6], 'tensor(float)')]


## 1. Zgodność ONNX ze sklearn

In [3]:
in_name = session.get_inputs()[0].name
label_onnx, proba_onnx = session.run(None, {in_name: X_te})
label_skl, proba_skl = model.predict(X_te), model.predict_proba(X_te)
parity = pd.Series({
    "dłonie w teście": len(X_te),
    "ta sama klasa [%]": (label_onnx == label_skl).mean() * 100,
    "max różnica prawdopodobieństw": float(np.abs(proba_onnx - proba_skl).max()),
})
parity.to_frame("wartość")

,wartość
dłonie w teście,127414.000000
ta sama klasa [%],100.000000
max różnica prawdopodobieństw,0.000002


## 2. Rozmiar i szybkość

In [4]:
x1 = X_te[:1]
for _ in range(100):
    session.run(None, {in_name: x1})
t0 = time.perf_counter()
for _ in range(5000):
    session.run(None, {in_name: x1})
onnx_ms = (time.perf_counter() - t0) / 5000 * 1000
pd.Series({"rozmiar ONNX [KB]": onnx_path.stat().st_size / 1024, "rozmiar pickle sklearn [KB]": gates["final"]["model_size_kb"],
           "1 dłoń, ONNX Runtime na laptopie [ms]": onnx_ms,
           "1 dłoń, sklearn na laptopie [ms]": gates["final"]["latency_ms_sklearn"]}).round(4).to_frame("wartość")

,wartość
rozmiar ONNX [KB],57.0059
rozmiar pickle sklearn [KB],175.2373
"1 dłoń, ONNX Runtime na laptopie [ms]",0.0097
"1 dłoń, sklearn na laptopie [ms]",0.1403


Telefon ma słabszy procesor niż laptop, ale nawet 10x wolniej to wciąż ułamek milisekundy na dłoń. Wąskim gardłem
na telefonie będzie MediaPipe (wykrycie dłoni i punktów), a nie nasz model.

## 3. Wektory testowe dla aplikacji

In [5]:
rng = np.random.default_rng(0)
pick = np.concatenate([rng.choice(np.where(y_te == c)[0], 3, replace=False) for c in range(6)])
vectors = [{"raw_landmarks_xy": lm_te[i].round(6).tolist(), "features": X_te[i].round(6).tolist(),
            "expected_label": int(label_onnx[i]), "expected_probabilities": proba_onnx[i].round(6).tolist(),
            "true_label": int(y_te[i])} for i in pick]
(DEPLOY / "test_vectors.json").write_text(json.dumps(vectors, indent=1))
spec = {
    "model_file": onnx_path.name, "input_name": in_name, "input_shape": [1, X_te.shape[1]], "input_dtype": "float32",
    "outputs": {o[0]: o[1] for o in outputs}, "classes": [int(c) for c in model.classes_],
    "preprocess": prep_name,
    "feature_order": "x0, y0, x1, y1, ..., x20, y20 (punkty MediaPipe, przeplecione)",
    "test_macro_f1": gates["final"]["test_macro_f1"], "test_stress_macro_f1": gates["final"]["test_stress_macro_f1"],
}
(DEPLOY / "model_spec.json").write_text(json.dumps(spec, indent=2, ensure_ascii=False))
print(json.dumps(spec, indent=2, ensure_ascii=False))

{
  "model_file": "finger_counter.onnx",
  "input_name": "X",
  "input_shape": [
    1,
    42
  ],
  "input_dtype": "float32",
  "outputs": {
    "label": [
      null
    ],
    "probabilities": [
      null,
      6
    ]
  },
  "classes": [
    0,
    1,
    2,
    3,
    4,
    5
  ],
  "preprocess": "wrist_scale",
  "feature_order": "x0, y0, x1, y1, ..., x20, y20 (punkty MediaPipe, przeplecione)",
  "test_macro_f1": 0.9942889371813476,
  "test_stress_macro_f1": 0.9903399029073113
}


## Wnioski

- **ONNX daje dokładnie to samo co sklearn.** Na 127 414 dłoniach testu klasy zgadzają się w 100%,
  a prawdopodobieństwa różnią się najwyżej o 2e-6 (dokładność float32).
- **Plik modelu ma 57 KB** zamiast 175 KB (pickle sklearn). Aplikacja go praktycznie nie odczuje.
- **Predykcja jednej dłoni trwa 0.01 ms w ONNX Runtime na laptopie,** 14 razy szybciej niż sklearn.
  Na telefonie będzie wolniej, ale przy 30 klatkach na sekundę mamy 33 ms na klatkę. Czas zajmie głównie MediaPipe,
  nie nasz model.
- `test_vectors.json` pozwala sprawdzić w aplikacji, że preprocessing w Kotlinie i model dają to samo co Python.
  Bez tego błąd w kolejności cech (x0, y0, x1, ... zamiast x0, x1, ...) dałby śmieci bez żadnego komunikatu.

**Następny krok:** aplikacja na Androida (krok 11 i 12 diagramu).